# Category channels analytics (PG → pandas)

Load `channel` ⋈ `channel_stat` for one category, keep top-N by latest `pv_score_rank`, plot period views for top-M.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import create_engine, text

# notebook lives in <repo>/analysis → yt_fetcher is sibling
ROOT = Path.cwd().resolve()
if ROOT.name == "analysis":
    YT = ROOT.parent / "yt_fetcher"
else:
    YT = ROOT / "yt_fetcher" if (ROOT / "yt_fetcher").exists() else ROOT

os.chdir(YT)  # so Settings picks up yt_fetcher/.env
sys.path.insert(0, str(YT))

from app.config import settings

engine = create_engine(
    f"postgresql+psycopg2://{settings.DB_USER}:{settings.DB_PASS}"
    f"@{settings.DB_HOST}:{settings.DB_PORT}/{settings.DB_NAME}"
)
print(f"PG {settings.DB_HOST}:{settings.DB_PORT}/{settings.DB_NAME}")

In [ ]:
CATEGORY_ID = 1
TOP_N = 50   # channels to load (by latest pv_score_rank)


SQL = text("""
WITH ranked AS (
    SELECT
        cs.channel_id,
        cs.pv_score_rank,
        ROW_NUMBER() OVER (
            ORDER BY cs.pv_score_rank NULLS LAST, cs.pv_view DESC NULLS LAST
        ) AS rn
    FROM channel_stat cs
    JOIN channel c ON c.channel_id = cs.channel_id
    WHERE c.category_id = :category_id
      AND c.status > 0
      AND cs.report_period = (
          SELECT MAX(cs2.report_period)
          FROM channel_stat cs2
          JOIN channel c2 ON c2.channel_id = cs2.channel_id
          WHERE c2.category_id = :category_id
            AND c2.status > 0
            AND cs2.pv_score_rank IS NOT NULL
      )
),
top_channels AS (
    SELECT channel_id
    FROM ranked
    WHERE rn <= :top_n
)
SELECT
    c.channel_id,
    c.channel_title,
    c.custom_url,
    c.category_id,
    c.status,
    c.priority,
    cs.report_period,
    cs.data_at,
    cs.subscriber_count,
    cs.channel_view_count,
    cs.video_count,
    cs.pc_view,
    cs.pc_subscriber,
    cs.pc_video,
    cs.pv_view,
    cs.pv_view_new_long,
    cs.pv_view_new_short,
    cs.pv_view_old_long,
    cs.pv_view_old_short,
    cs.pv_like,
    cs.pv_comment,
    cs.pv_score,
    cs.pv_score_rank,
    cs.pv_score_change,
    cs.pv_score_rank_change,
    cs.pv_video_long,
    cs.pv_video_short,
    cs.pv_duration
FROM channel_stat cs
JOIN channel c ON c.channel_id = cs.channel_id
JOIN top_channels t ON t.channel_id = c.channel_id
WHERE c.category_id = :category_id
ORDER BY cs.report_period, cs.pv_score_rank NULLS LAST
""")

df = pd.read_sql(
    SQL,
    engine,
    params={"category_id": CATEGORY_ID, "top_n": TOP_N},
    parse_dates=["report_period", "data_at"],
)
print(df.shape)
print("periods:", sorted(df["report_period"].dropna().unique())[:3], "…",
      sorted(df["report_period"].dropna().unique())[-3:])
print("channels:", df["channel_id"].nunique())
df.head()

In [ ]:
TOP_M = 20   # channels whose pv_view is summed on the chart
# Top-M by latest-period rank (among already loaded top-N)
latest = df["report_period"].max()
top_m_ids = (
    df.loc[df["report_period"] == latest]
    .sort_values(["pv_score_rank", "pv_view"], ascending=[True, False])
    .head(TOP_M)["channel_id"]
    .tolist()
)

by_period = (
    df.loc[df["channel_id"].isin(top_m_ids)]
    .groupby("report_period", as_index=False)["pv_view"]
    .sum()
    .sort_values("report_period")
)
by_period["pv_view_m"] = (by_period["pv_view"] / 1e6).round(0).astype(int)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(by_period["report_period"], by_period["pv_view_m"], width=20, align="center")
ax.set_ylim(bottom=0)
ax.set_title(
    f"Cat {CATEGORY_ID}: sum(pv_view) of top-{TOP_M} channels by period"
)
ax.set_xlabel("report_period")
ax.set_ylabel("pv_view, millions")
ax.grid(True, axis="y", alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

by_period[["report_period", "pv_view_m"]]